# 82 - Seguimiento múltiple (mini‑SORT) — Base

In [ ]:
import numpy as np
from dataclasses import dataclass

def iou(a,b):
    xA=max(a[0],b[0]); yA=max(a[1],b[1]); xB=min(a[2],b[2]); yB=min(a[3],b[3])
    inter=max(0,xB-xA)*max(0,yB-yA)
    ua=(a[2]-a[0])*(a[3]-a[1]); ub=(b[2]-b[0])*(b[3]-b[1])
    union=ua+ub-inter
    return inter/union if union>0 else 0.0

@dataclass
class Track:
    id:int; box:np.ndarray; hits:int=0; age:int=0; time_since_update:int=0

class MiniSORT:
    def __init__(self,iou_thresh=0.3,max_age=5):
        self.iou_thresh=iou_thresh; self.max_age=max_age; self.next_id=1; self.tracks=[]
    def update(self,dets):
        assigned=set()
        for tr in self.tracks:
            tr.age+=1; tr.time_since_update+=1
            best=-1; bests=0
            for j,d in enumerate(dets):
                if j in assigned: continue
                s=iou(tr.box,d)
                if s>bests: bests=s; best=j
            if bests>=self.iou_thresh:
                tr.box=0.8*tr.box+0.2*dets[best]
                tr.hits+=1; tr.time_since_update=0; assigned.add(best)
        for j,d in enumerate(dets):
            if j not in assigned:
                self.tracks.append(Track(self.next_id,d.copy(),hits=1)); self.next_id+=1
        self.tracks=[t for t in self.tracks if t.time_since_update<=self.max_age]
        return [(t.id,t.box.copy()) for t in self.tracks]

rng=np.random.default_rng(0); tracker=MiniSORT(0.2,3); traj={}
for t in range(15):
    dets=[np.array([10+2*t,20,40+2*t,50],float), np.array([200-3*t,150,230-3*t,180],float)]
    if rng.random()<0.2: dets.append(np.array([100,100,120,120],float))
    tr=tracker.update(dets)
    for tid,box in tr: traj.setdefault(tid,[]).append((t,box.tolist()))
print('IDs:',sorted(traj.keys()))
